# Objetivo B - Modelo Federado Para Inferencia En Banco Guatemala

Este notebook implementa el flujo del Objetivo B del proyecto PlusTI.

Se utilizarán los bancos etiquetados:
- Bolivia
- Brazil

Y se generarán inferencias para:
- Guatemala

La metodología reutiliza aprendizajes del Objetivo A, pero entrena un modelo nuevo con Bolivia + Brazil.

In [1]:
import os
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

os.makedirs("resultados_objetivo_b", exist_ok=True)

print("Librerías cargadas.")
print("Carpeta resultados_objetivo_b/ lista.")

Librerías cargadas.
Carpeta resultados_objetivo_b/ lista.


## 1. Definir Rutas

In [2]:
PATHS = {
    "bolivia_limpio": "data_limpia/bolivia_limpio.csv",
    "brazil_limpio": "data_limpia/brazil_limpio.csv",
    "guatemala_limpio": "data_limpia/guatemala_limpio.csv",
    "bolivia_model": "data_limpia/bolivia_model_ready.csv",
    "brazil_model": "data_limpia/brazil_model_ready.csv",
    "guatemala_model": "data_limpia/guatemala_model_ready.csv",
}

for name, path in PATHS.items():
    print(name, "->", path, "| existe:", os.path.exists(path))

bolivia_limpio -> data_limpia/bolivia_limpio.csv | existe: True
brazil_limpio -> data_limpia/brazil_limpio.csv | existe: True
guatemala_limpio -> data_limpia/guatemala_limpio.csv | existe: True
bolivia_model -> data_limpia/bolivia_model_ready.csv | existe: True
brazil_model -> data_limpia/brazil_model_ready.csv | existe: True
guatemala_model -> data_limpia/guatemala_model_ready.csv | existe: True


## 2. Cargar Datasets

In [3]:
bolivia_limpio = pd.read_csv(PATHS["bolivia_limpio"], low_memory=False)
brazil_limpio = pd.read_csv(PATHS["brazil_limpio"], low_memory=False)
guatemala_limpio = pd.read_csv(PATHS["guatemala_limpio"], low_memory=False)

bolivia_model = pd.read_csv(PATHS["bolivia_model"], low_memory=False)
brazil_model = pd.read_csv(PATHS["brazil_model"], low_memory=False)
guatemala_model = pd.read_csv(PATHS["guatemala_model"], low_memory=False)

datasets = {
    "bolivia_limpio": bolivia_limpio,
    "brazil_limpio": brazil_limpio,
    "guatemala_limpio": guatemala_limpio,
    "bolivia_model": bolivia_model,
    "brazil_model": brazil_model,
    "guatemala_model": guatemala_model,
}

## 3. Validar Dimensiones

In [4]:
resumen_dimensiones = pd.DataFrame([
    {
        "dataset": name,
        "filas": df.shape[0],
        "columnas": df.shape[1],
    }
    for name, df in datasets.items()
])

resumen_dimensiones

,dataset,filas,columnas
0,bolivia_limpio,100003,71
1,brazil_limpio,100000,71
2,guatemala_limpio,100000,69
3,bolivia_model,100003,51
4,brazil_model,100000,52
5,guatemala_model,100000,50


## 4. Validar Targets

In [5]:
print("Bolivia target:")
print(bolivia_model["is_fraud_binary"].value_counts(dropna=False))

print("\nBrazil target:")
print(brazil_model["is_fraud_binary"].value_counts(dropna=False))

print("\nGuatemala columnas target:")
print([c for c in guatemala_model.columns if "fraud" in c.lower()])

Bolivia target:
is_fraud_binary
0    95084
1     4919
Name: count, dtype: int64

Brazil target:
is_fraud_binary
0    96795
1     3205
Name: count, dtype: int64

Guatemala columnas target:
[]


## 5. Revisar Columnas Comunes

In [6]:
cols_bolivia = set(bolivia_model.columns)
cols_brazil = set(brazil_model.columns)
cols_guatemala = set(guatemala_model.columns)

cols_comunes = sorted(cols_bolivia & cols_brazil & cols_guatemala)
cols_union = sorted(cols_bolivia | cols_brazil | cols_guatemala)

print("Columnas Bolivia:", len(cols_bolivia))
print("Columnas Brazil:", len(cols_brazil))
print("Columnas Guatemala:", len(cols_guatemala))
print("Columnas comunes:", len(cols_comunes))
print("Columnas unión:", len(cols_union))

print("\nColumnas en Brazil pero no en Bolivia:")
print(sorted(cols_brazil - cols_bolivia))

print("\nColumnas en Guatemala pero no en Bolivia:")
print(sorted(cols_guatemala - cols_bolivia))

print("\nColumnas en Bolivia pero no en Guatemala:")
print(sorted(cols_bolivia - cols_guatemala))

Columnas Bolivia: 51
Columnas Brazil: 52
Columnas Guatemala: 50
Columnas comunes: 49
Columnas unión: 52

Columnas en Brazil pero no en Bolivia:
['DE2_PAN']

Columnas en Guatemala pero no en Bolivia:
['DE2_PAN']

Columnas en Bolivia pero no en Guatemala:
['is_fraud', 'is_fraud_binary']


## 6. Definir Features Base Seguras

In [7]:
TARGET_COLS = ["is_fraud", "is_fraud_binary"]

LOCAL_OR_SENSITIVE_COLUMNS = [
    "bank_code",
    "bank_name",
    "bank_country",
    "bank_tier",
    "transaction_id",
    "client_id",
    "pan_masked",
    "pan_hash",
    "DE2_PAN",
    "DE35_track2_data_masked",
    "DE37_retrieval_reference_number",
    "DE38_authorization_code",
    "DE41_terminal_id",
    "DE42_card_acceptor_id",
    "DE102_account_id_1",
    "DE103_account_id_2",
]

feature_cols_base = sorted([
    col for col in cols_comunes
    if col not in TARGET_COLS
    and col not in LOCAL_OR_SENSITIVE_COLUMNS
])

print("Cantidad de features base:", len(feature_cols_base))
print("Features base:")
for col in feature_cols_base:
    print("-", col)

Cantidad de features base: 49
Features base:
- DE100_receiving_institution_id
- DE11_STAN
- DE123_pos_data_code
- DE12_local_time
- DE13_local_date
- DE14_expiration_date
- DE15_settlement_date
- DE18_merchant_category_code
- DE19_acquirer_country_code
- DE22_pos_entry_mode
- DE23_card_seq_number
- DE25_pos_condition_code
- DE39_response_code
- DE3_processing_code
- DE43_card_acceptor_name_location
- DE49_currency_code_transaction
- DE4_amount_transaction
- DE52_pin_data_present
- DE55_emv_data_present
- DE58_authorizing_agent_id
- DE60_pos_terminal_type
- DE61_pos_extended_data
- DE63_network_specific
- DE6_amount_cardholder_billing
- DE7_transmission_datetime
- DE9_conversion_rate_billing
- amount_diff_baseline
- amount_local
- amount_tx_currency
- amount_usd
- amount_vs_baseline
- approved
- card_brand
- channel
- client_baseline_amount
- client_home_city
- client_segment
- currency_tx_alpha
- day_of_week
- distance_from_home_km
- distance_from_home_km_was_missing
- high_amount_p95


In [8]:
X_bo_base = bolivia_model[feature_cols_base].copy()
X_br_base = brazil_model[feature_cols_base].copy()
X_gt_base = guatemala_model[feature_cols_base].copy()

print("Bolivia X base:", X_bo_base.shape)
print("Brazil X base:", X_br_base.shape)
print("Guatemala X base:", X_gt_base.shape)

assert list(X_bo_base.columns) == list(X_br_base.columns) == list(X_gt_base.columns)

print("Columnas alineadas correctamente.")

Bolivia X base: (100003, 49)
Brazil X base: (100000, 49)
Guatemala X base: (100000, 49)
Columnas alineadas correctamente.


In [9]:
resumen_features = pd.DataFrame({
    "feature": feature_cols_base,
    "dtype_bolivia": [str(X_bo_base[c].dtype) for c in feature_cols_base],
    "dtype_brazil": [str(X_br_base[c].dtype) for c in feature_cols_base],
    "dtype_guatemala": [str(X_gt_base[c].dtype) for c in feature_cols_base],
    "nulos_bolivia": [X_bo_base[c].isna().sum() for c in feature_cols_base],
    "nulos_brazil": [X_br_base[c].isna().sum() for c in feature_cols_base],
    "nulos_guatemala": [X_gt_base[c].isna().sum() for c in feature_cols_base],
    "unicos_bolivia": [X_bo_base[c].nunique(dropna=True) for c in feature_cols_base],
    "unicos_brazil": [X_br_base[c].nunique(dropna=True) for c in feature_cols_base],
    "unicos_guatemala": [X_gt_base[c].nunique(dropna=True) for c in feature_cols_base],
})

resumen_features

,feature,dtype_bolivia,dtype_brazil,dtype_guatemala,nulos_bolivia,nulos_brazil,nulos_guatemala,unicos_bolivia,unicos_brazil,unicos_guatemala
0,DE100_receiving_institution_id,float64,float64,float64,1050,1010,2983,1,1,1
1,DE11_STAN,int64,int64,int64,0,0,0,100003,100000,100000
2,DE123_pos_data_code,object,object,object,954,989,3090,8,8,8
3,DE12_local_time,int64,int64,int64,0,0,0,56346,56575,56467
4,DE13_local_date,int64,int64,int64,0,0,0,181,182,183
5,DE14_expiration_date,int64,int64,int64,0,0,0,71,70,71
6,DE15_settlement_date,float64,float64,float64,982,1025,2935,181,182,183
7,DE18_merchant_category_code,int64,int64,int64,0,0,0,26,26,26
8,DE19_acquirer_country_code,int64,int64,int64,0,0,0,8,8,8
9,DE22_pos_entry_mode,int64,int64,int64,0,0,0,7,7,7


## 7. Ingeniería Temporal Reutilizable

In [10]:
def parse_iso8583_datetime(series, year=2025):
    """
    Convierte DE7_transmission_datetime desde formato MMDDHHMMSS numérico
    hacia datetime real usando el año del dataset.
    """
    return pd.to_datetime(
        str(year) + series.astype(str).str.zfill(10),
        format="%Y%m%d%H%M%S",
        errors="coerce"
    )

In [11]:
TEMPORAL_FEATURES = [
    "time_since_last_txn_min",
    "txn_count_last_1h",
    "txn_count_last_24h",
    "amount_zscore_customer",
]


def add_temporal_features(df_limpio, train_months=(1, 2, 3, 4, 5), year=2025):
    """
    Crea features temporales por cliente usando el orden cronológico de transacciones.

    Requiere columnas:
    - DE7_transmission_datetime
    - client_id
    - amount_usd

    Retorna un DataFrame con:
    - txn_dt
    - time_since_last_txn_min
    - txn_count_last_1h
    - txn_count_last_24h
    - amount_zscore_customer
    """
    required_cols = ["DE7_transmission_datetime", "client_id", "amount_usd"]
    missing = [c for c in required_cols if c not in df_limpio.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    df_temp = df_limpio[required_cols].copy()

    df_temp["txn_dt"] = parse_iso8583_datetime(
        df_temp["DE7_transmission_datetime"],
        year=year
    )

    if df_temp["txn_dt"].isna().any():
        n_bad = df_temp["txn_dt"].isna().sum()
        raise ValueError(f"No se pudieron parsear {n_bad} timestamps.")

    df_temp["amount_usd"] = pd.to_numeric(df_temp["amount_usd"], errors="coerce")
    df_temp["amount_usd"] = df_temp["amount_usd"].fillna(df_temp["amount_usd"].median())

    sorted_df = df_temp.sort_values(["client_id", "txn_dt"]).copy()

    sorted_df["time_since_last_txn_min"] = (
        sorted_df.groupby("client_id")["txn_dt"]
        .diff()
        .dt.total_seconds()
        .div(60)
    )

    median_time = sorted_df["time_since_last_txn_min"].median()
    sorted_df["time_since_last_txn_min"] = (
        sorted_df["time_since_last_txn_min"].fillna(median_time)
    )

    counts_1h = []
    counts_24h = []

    for client_id, group in sorted_df.groupby("client_id", sort=False):
        g = group.set_index("txn_dt")

        count_1h = (
            g["amount_usd"]
            .rolling("60min", closed="left")
            .count()
            .fillna(0)
            .astype(int)
        )

        count_24h = (
            g["amount_usd"]
            .rolling("24H", closed="left")
            .count()
            .fillna(0)
            .astype(int)
        )

        counts_1h.append(pd.Series(count_1h.values, index=group.index))
        counts_24h.append(pd.Series(count_24h.values, index=group.index))

    sorted_df["txn_count_last_1h"] = pd.concat(counts_1h).sort_index()
    sorted_df["txn_count_last_24h"] = pd.concat(counts_24h).sort_index()

    is_train_time = sorted_df["txn_dt"].dt.month.isin(train_months)

    client_stats = (
        sorted_df.loc[is_train_time]
        .groupby("client_id")["amount_usd"]
        .agg(client_mean="mean", client_std="std")
    )

    sorted_df = sorted_df.join(client_stats, on="client_id")

    global_mean = sorted_df.loc[is_train_time, "amount_usd"].mean()
    global_std = sorted_df.loc[is_train_time, "amount_usd"].std()

    sorted_df["client_mean"] = sorted_df["client_mean"].fillna(global_mean)
    sorted_df["client_std"] = sorted_df["client_std"].fillna(global_std)
    sorted_df["client_std"] = sorted_df["client_std"].replace(0, global_std)

    sorted_df["amount_zscore_customer"] = (
        (sorted_df["amount_usd"] - sorted_df["client_mean"])
        / (sorted_df["client_std"] + 1e-10)
    )

    temporal_df = (
        sorted_df
        .sort_index()
        [["txn_dt"] + TEMPORAL_FEATURES]
        .copy()
    )

    return temporal_df

In [12]:
temporal_bo = add_temporal_features(bolivia_limpio)
temporal_br = add_temporal_features(brazil_limpio)
temporal_gt = add_temporal_features(guatemala_limpio)

print("Temporal Bolivia:", temporal_bo.shape)
print("Temporal Brazil:", temporal_br.shape)
print("Temporal Guatemala:", temporal_gt.shape)

display(temporal_bo.head())

Temporal Bolivia: (100003, 5)
Temporal Brazil: (100000, 5)
Temporal Guatemala: (100000, 5)


,txn_dt,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer
0,2025-01-01 00:01:51,6854.0500,0,0,0.0730
1,2025-01-01 00:03:55,6854.0500,0,0,2.7514
2,2025-01-01 00:04:10,6854.0500,0,0,-0.2469
3,2025-01-01 00:04:53,6854.0500,0,0,-0.6362
4,2025-01-01 00:07:56,6854.0500,0,0,-0.7176


In [13]:
for name, temporal_df in {
    "Bolivia": temporal_bo,
    "Brazil": temporal_br,
    "Guatemala": temporal_gt,
}.items():
    print(f"\n{name}")
    print("Rango fechas:", temporal_df["txn_dt"].min(), "->", temporal_df["txn_dt"].max())
    print("Nulos txn_dt:", temporal_df["txn_dt"].isna().sum())
    print("Distribución meses:")
    print(temporal_df["txn_dt"].dt.month.value_counts().sort_index())


Bolivia
Rango fechas: 2025-01-01 00:01:51 -> 2025-06-30 00:13:36
Nulos txn_dt: 0
Distribución meses:
txn_dt
1    17236
2    15663
3    17130
4    16615
5    17173
6    16186
Name: count, dtype: int64

Brazil
Rango fechas: 2025-01-01 00:02:21 -> 2025-07-01 01:57:51
Nulos txn_dt: 0
Distribución meses:
txn_dt
1    17160
2    15388
3    17314
4    16752
5    17318
6    16067
7        1
Name: count, dtype: int64

Guatemala
Rango fechas: 2025-01-01 00:06:28 -> 2025-07-01 15:22:18
Nulos txn_dt: 0
Distribución meses:
txn_dt
1    17345
2    15335
3    17265
4    16843
5    17131
6    16080
7        1
Name: count, dtype: int64


In [14]:
for name, temporal_df in {
    "Bolivia": temporal_bo,
    "Brazil": temporal_br,
    "Guatemala": temporal_gt,
}.items():
    print(f"\n{name}")
    print(temporal_df[TEMPORAL_FEATURES].isna().sum())
    display(temporal_df[TEMPORAL_FEATURES].describe().round(2))


Bolivia
time_since_last_txn_min    0
txn_count_last_1h          0
txn_count_last_24h         0
amount_zscore_customer     0
dtype: int64


,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer
count,100003.0000,100003.0000,100003.0000,100003.0000
mean,9798.4100,0.0600,0.2100,0.0200
std,9985.0300,0.4200,0.6200,1.0700
min,0.1700,0.0000,0.0000,-1.8400
25%,2772.6600,0.0000,0.0000,-0.6300
50%,6854.0500,0.0000,0.0000,-0.3900
75%,13456.4500,0.0000,0.0000,0.2100
max,126570.5200,7.0000,9.0000,39.4600



Brazil
time_since_last_txn_min    0
txn_count_last_1h          0
txn_count_last_24h         0
amount_zscore_customer     0
dtype: int64


,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer
count,100000.0000,100000.0000,100000.0000,100000.0000
mean,9802.0000,0.0100,0.1500,0.0300
std,9762.1000,0.0900,0.4000,1.1400
min,0.1000,0.0000,0.0000,-2.6300
25%,2893.4700,0.0000,0.0000,-0.5700
50%,6900.4200,0.0000,0.0000,-0.3100
75%,13442.8000,0.0000,0.0000,0.1700
max,114810.7200,2.0000,5.0000,30.0300



Guatemala
time_since_last_txn_min    0
txn_count_last_1h          0
txn_count_last_24h         0
amount_zscore_customer     0
dtype: int64


,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer
count,100000.0000,100000.0000,100000.0000,100000.0000
mean,12059.5200,0.0200,0.1500,0.0500
std,12219.6100,0.2600,0.4800,1.3500
min,0.0200,0.0000,0.0000,-2.8300
25%,3517.3800,0.0000,0.0000,-0.5800
50%,8430.9500,0.0000,0.0000,-0.2800
75%,16481.3100,0.0000,0.0000,0.2600
max,149597.7700,6.0000,7.0000,59.8800


In [15]:
X_bo_full = X_bo_base.join(temporal_bo[TEMPORAL_FEATURES])
X_br_full = X_br_base.join(temporal_br[TEMPORAL_FEATURES])
X_gt_full = X_gt_base.join(temporal_gt[TEMPORAL_FEATURES])

print("X Bolivia full:", X_bo_full.shape)
print("X Brazil full:", X_br_full.shape)
print("X Guatemala full:", X_gt_full.shape)

assert list(X_bo_full.columns) == list(X_br_full.columns) == list(X_gt_full.columns)

print("Features base + temporales alineadas correctamente.")

X Bolivia full: (100003, 53)
X Brazil full: (100000, 53)
X Guatemala full: (100000, 53)
Features base + temporales alineadas correctamente.


In [16]:
feature_cols_full = X_bo_full.columns.tolist()

print("Cantidad total de features:", len(feature_cols_full))
print("Features temporales agregadas:")
print(TEMPORAL_FEATURES)

Cantidad total de features: 53
Features temporales agregadas:
['time_since_last_txn_min', 'txn_count_last_1h', 'txn_count_last_24h', 'amount_zscore_customer']
